In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        #"temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [8]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [10]:
def run_propmt(test_case):
    """ Merges the prompt with the test case and returns the result """
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    return text


In [12]:
def run_test_case(test_case):
    """Calls run_propmt, then grades the result"""
    output = run_propmt(test_case)
    # todo - grading

    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

In [13]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [14]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


In [15]:
print(results)

[{'output': '# AWS S3 Region Extraction Function\n\nHere\'s a comprehensive solution to extract the AWS region from an S3 bucket URI:\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\nfrom typing import Optional\nimport re\n\ndef extract_region_from_s3_uri(s3_uri: str) -> Optional[str]:\n    """\n    Extracts the AWS region from an S3 bucket URI by querying the bucket\'s location constraint.\n    \n    Args:\n        s3_uri (str): S3 URI in the format \'s3://bucket-name/key\' or \'s3://bucket-name\'\n        \n    Returns:\n        Optional[str]: AWS region name (e.g., \'us-east-1\'), or None if extraction fails\n        \n    Raises:\n        ValueError: If the URI format is invalid\n        ClientError: If AWS API call fails\n    """\n    # Validate and parse the S3 URI\n    s3_pattern = r\'^s3://([a-z0-9.-]+)(?:/.*)?$\'\n    match = re.match(s3_pattern, s3_uri)\n    \n    if not match:\n        raise ValueError(f"Invalid S3 URI format: {s3_uri}. Expected forma